Importing Tools

In [45]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib

Generating synthetic training data

In [46]:
# Set random seed for reproducibility
np.random.seed(42)

In [47]:
# Generate 1 year of hourly timestamps
dates = pd.date_range(start="2024-01-01", end="2024-12-31 23:00:00", freq="h")
n = len(dates)

hour = dates.hour
month = dates.month

In [48]:
# Simulate solar irradiance based on time of day (W/m2)
solar_radiation = np.maximum(0, np.sin((hour - 6) * np.pi /12 * 1000)) + np.random.normal(0, 50, n)
solar_radiation = np.clip(solar_radiation, 0, 1200)

In [49]:
# Simulate ambient temperature (°C) and cloud cover (%)
temperature = 15 + 15 * np.sin((hour - 9) * np.pi / 12) + np.random.normal(0, 3, n)
cloud_cover = np.random.uniform(0, 100, n)

In [50]:
ac_power = (solar_radiation * 25.0) * (1 - (cloud_cover / 100) * 0.4) - (temperature * 2.0)
ac_power = np.maximum(0, ac_power)

Formatting the dataset

In [51]:
df = pd.DataFrame({
    'datetime': dates,
    'hour': hour,
    'month': month,
    'solar_radiation': solar_radiation,
    'temperature': temperature,
    'cloud_cover': cloud_cover,
    'ac_power_kw': ac_power

})

Splitting data for training

In [52]:
X = df[['hour', 'month', 'solar_radiation', 'temperature', 'cloud_cover']]
y = df[['ac_power_kw']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Model Training & Evaluation

In [53]:
# Train XGBoost Regressor
model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [54]:
# Evaluate model performance
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Model Training Complete!")
print(f" - Root Mean Squared Error (RMSE): {rmse:.2f} kW")
print(f" - R² Accuracy Score: {r2:.4f}")

Model Training Complete!
 - Root Mean Squared Error (RMSE): 27.39 kW
 - R² Accuracy Score: 0.9979


In [55]:
# Save model
joblib.dump(model, "solar_xgb_model.pkl")
print("Model exported as 'solar_xgb_model.pkl'")

Model exported as 'solar_xgb_model.pkl'
